<a href="https://colab.research.google.com/github/AnzorGozalishvili/IOAI-2025-lectures/blob/main/team_training_session/workshop_1/HPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Hyperparameter Optimization (HPO) Workshop – IOAI 2025

Welcome! In this workshop, you'll learn how to tune hyperparameters effectively for both classical machine learning and deep learning models using **Optuna** and other medium-level tools. This notebook is structured into:

- Intro to HPO
- HPO for Classical ML Models
- HPO for Deep Learning Models
- Best Practices and Recommendations


## 🏁 1. What is HPO?

**Hyperparameter Optimization (HPO)** is the process of searching for the best set of model configurations (hyperparameters) that yield the highest model performance.

### 🔧 What are hyperparameters?
- Parameters set **before** training (e.g., learning rate, depth of tree).
- Not learned during the training process.

### 🎯 What does HPO do?
- Defines a **search space** and **objective function**.
- Tries different configurations to find the best performance (accuracy, loss, etc.).

### ✅ Where is it used?
- Classical ML (SVM, Random Forests, etc.)
- Deep Learning (NNs, CNNs, Transformers)


## 🌡️ Warm-Up Notes

- **HPO is resource-heavy** for large models.
- **Classical ML** models have many hyperparams to tune.
- **DL models are more robust**, often need fewer tunings.
- **DL tuning is data-related**, classical ML is more model-related.
- **Classical ML needs full training per trial**, no intermediate evaluation.


## 🗺️ 2. HPO on Classical ML Models

### Steps:

1. Identify hyperparameters
2. Select important ones
3. Define search strategy
4. Use HPO tools (Optuna, scikit-optimize)
5. Validation strategy (K-Fold, holdout)
6. Run optimization
7. Analyze results
8. Pick the best model


In [3]:
# !pip install optuna

In [4]:
# HPO on Classical ML with Optuna and RandomForest
import optuna
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

# Load dataset
X, y = load_iris(return_X_y=True)

# Define the objective function for optimization
def objective(trial):
    clf = RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 10, 200),
        max_depth=trial.suggest_int("max_depth", 2, 20),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
        random_state=42
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    return cross_val_score(clf, X, y, cv=cv).mean()

# Run optimization
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

# Show the best trial
print("Best trial parameters:", study.best_trial.params)


[I 2025-06-30 13:48:37,055] A new study created in memory with name: no-name-fdedd30b-5b2a-44e9-9db8-166249bc993f
[I 2025-06-30 13:48:37,618] Trial 0 finished with value: 0.9466666666666667 and parameters: {'n_estimators': 84, 'max_depth': 11, 'min_samples_split': 2}. Best is trial 0 with value: 0.9466666666666667.
[I 2025-06-30 13:48:38,229] Trial 1 finished with value: 0.9666666666666668 and parameters: {'n_estimators': 96, 'max_depth': 16, 'min_samples_split': 7}. Best is trial 1 with value: 0.9666666666666668.
[I 2025-06-30 13:48:39,662] Trial 2 finished with value: 0.9666666666666668 and parameters: {'n_estimators': 180, 'max_depth': 11, 'min_samples_split': 13}. Best is trial 1 with value: 0.9666666666666668.
[I 2025-06-30 13:48:40,281] Trial 3 finished with value: 0.9600000000000002 and parameters: {'n_estimators': 94, 'max_depth': 3, 'min_samples_split': 15}. Best is trial 1 with value: 0.9666666666666668.
[I 2025-06-30 13:48:40,939] Trial 4 finished with value: 0.9666666666666

Best trial parameters: {'n_estimators': 96, 'max_depth': 16, 'min_samples_split': 7}


## 🤖 3. HPO on Deep Learning Models

### Steps:

1. Identify hyperparameters (batch size, LR, dropout, etc.)
2. Select impactful ones
3. Use proper search strategy
4. Tools: Optuna, Ray Tune, etc.
5. Define validation
6. Run HPO loop with early stopping
7. Analyze results
8. Retrain best configuration


In [5]:
# HPO on Deep Learning with Optuna and PyTorch (MNIST)
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import optuna

class SimpleNet(nn.Module):
    def __init__(self, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.model(x)

def objective(trial):
    lr = trial.suggest_loguniform("lr", 1e-4, 1e-1)
    dropout = trial.suggest_float("dropout", 0.2, 0.5)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    transform = transforms.ToTensor()
    train_dataset = datasets.MNIST('.', train=True, download=True, transform=transform)
    train_len = int(len(train_dataset) * 0.8)
    val_len = len(train_dataset) - train_len
    train_data, val_data = random_split(train_dataset, [train_len, val_len])
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=128)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SimpleNet(dropout).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(5):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            optimizer.step()

    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()

    return correct / len(val_loader.dataset)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print("Best hyperparameters:", study.best_trial.params)


[I 2025-06-30 13:53:24,403] A new study created in memory with name: no-name-f618ab05-4e3a-48f6-92c8-be957108cc2f
/tmp/ipython-input-5-1670205410.py:23: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-4, 1e-1)
100%|██████████| 9.91M/9.91M [00:00<00:00, 15.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 477kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.81MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.89MB/s]
[I 2025-06-30 13:54:03,324] Trial 0 finished with value: 0.9704166666666667 and parameters: {'lr': 0.0017415670306833377, 'dropout': 0.2950575464302256, 'batch_size': 32}. Best is trial 0 with value: 0.9704166666666667.
[I 2025-06-30 13:54:28,564] Trial 1 finished with value: 0.6799166666666666 and parameters: {'lr': 0.0844669994709161, 'dropout': 0.43737883664755056, 'batch_siz

Best hyperparameters: {'lr': 0.0024358067027541144, 'dropout': 0.21451457228146642, 'batch_size': 32}


## 🛡️ Best Practices

- Avoid overfitting on validation set
- Set compute budgets (e.g., 30 trials, max 2 hours)
- Use pruning to stop bad trials early
- Fix random seeds for reproducibility
- Log and analyze performance for each trial
